&nbsp;
## Tokenizing text

In [1]:
import requests

url = (
    "https://raw.githubusercontent.com/rasbt/"
    "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
    "the-verdict.txt"
)
file_path = "the-verdict.txt"

response = requests.get(url, timeout=30)
response.raise_for_status()
with open(file_path, "wb") as f:
    f.write(response.content)

In [2]:
with open(file_path, "r", encoding="utf-8") as f:
    raw_text = f.read()
print("Total number of characters in the text:", len(raw_text))
print(raw_text[:99])

Total number of characters in the text: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [3]:
import re

preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(len(preprocessed))
print(preprocessed[:50])

4690
['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in', 'the', 'height', 'of', 'his', 'glory', ',', 'he', 'had', 'dropped', 'his', 'painting', ',', 'married', 'a', 'rich', 'widow', ',', 'and', 'established', 'himself']


## vocabulary 

In [4]:
all_words= sorted(set(preprocessed))
vocab_size= len(all_words)
print("Vocabulary size:", vocab_size)
print("First 20 words in the vocabulary:", all_words[:20])

Vocabulary size: 1130
First 20 words in the vocabulary: ['!', '"', "'", '(', ')', ',', '--', '.', ':', ';', '?', 'A', 'Ah', 'Among', 'And', 'Are', 'Arrt', 'As', 'At', 'Be']


In [5]:
vocab = {token: idx for idx, token in enumerate(all_words)}
for i,item in enumerate(vocab.items()):
    print(f"Token: '{item[0]}' -> Index: {item[1]}")
    if i >= 19:
        break


Token: '!' -> Index: 0
Token: '"' -> Index: 1
Token: ''' -> Index: 2
Token: '(' -> Index: 3
Token: ')' -> Index: 4
Token: ',' -> Index: 5
Token: '--' -> Index: 6
Token: '.' -> Index: 7
Token: ':' -> Index: 8
Token: ';' -> Index: 9
Token: '?' -> Index: 10
Token: 'A' -> Index: 11
Token: 'Ah' -> Index: 12
Token: 'Among' -> Index: 13
Token: 'And' -> Index: 14
Token: 'Are' -> Index: 15
Token: 'Arrt' -> Index: 16
Token: 'As' -> Index: 17
Token: 'At' -> Index: 18
Token: 'Be' -> Index: 19


## SimpleTokenizer Function 

### It used encode method and decode method



In [6]:
class SimpleTokenizer:
    def __init__(self,vocab):
        self.vocab = vocab
        self.inv_vocab = {idx: token for token, idx in vocab.items()}

    def encode(self,text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        idx = [self.vocab[token] for token in preprocessed]
        return idx
    
    def decode(self,idx):
        text = " ".join([self.inv_vocab[i] for i in idx])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [7]:
tokenizer = SimpleTokenizer(vocab)
text = """"It's the last he painted, you know," 
           Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)


[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [11]:
decoded_text = tokenizer.decode(ids)
print(decoded_text)

" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


In [56]:
# sample1 ="Hello, do you like tea?"
# print(tokenizer.encode(sample1))


## Adding special context tokens

In [19]:
all_words= sorted(set(preprocessed))
all_words.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token: idx for idx, token in enumerate(all_words)}
vocab_size= len(all_words)
print("Vocabulary size:", vocab_size)

Vocabulary size: 1132


In [23]:
for idx, item in enumerate(list(vocab.items())[-5:]):
    print(f"Token: '{item[0]}' -> Index: {item[1]}")
      


Token: 'younger' -> Index: 1127
Token: 'your' -> Index: 1128
Token: 'yourself' -> Index: 1129
Token: '<|endoftext|>' -> Index: 1130
Token: '<|unk|>' -> Index: 1131


In [25]:
class SimpleTokenizerV2:
    def __init__(self,vocab):
        self.vocab = vocab
        self.inv_vocab = {idx: token for token, idx in vocab.items()}

    def encode(self,text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [item if item in self.vocab else '<|unk|>' for item in preprocessed]
        idx = [self.vocab[token] for token in preprocessed]
        return idx
    
    def decode(self,idx):
        text = " ".join([self.inv_vocab[i] for i in idx])
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

In [26]:
sample1 ="Hello, do you like tea?"
sample2 ="In the sunlit terraces of the palace."
text = " <|endoftext|> ".join([sample1, sample2])
print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [27]:
tokenizer_v2 = SimpleTokenizerV2(vocab)
print(tokenizer_v2.encode(text))

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]


In [31]:
print(tokenizer_v2.decode(tokenizer_v2.encode(text)))

<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.


## Byte Pair Encoding

In [35]:
from importlib.metadata import version
import tiktoken
print("tiktoken version:", version("tiktoken"))

tiktoken version: 0.12.0


In [36]:
tokenizer_bpe = tiktoken.get_encoding("gpt2")

In [48]:
text = ("Hello, do you like tea? <|endoftext|> In the sunlit terraces"
         " of someunknownPlace.")
integers = tokenizer_bpe.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 286, 617, 34680, 27271, 13]


In [49]:
strings = tokenizer_bpe.decode(integers)
print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace.


In [50]:
text = "Akwirw ier"
integers = tokenizer_bpe.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

[33901, 86, 343, 86, 220, 959]


In [55]:
strings = tokenizer_bpe.decode(integers)
print(strings)

Akwirw ier


In [57]:
encoded_text = tokenizer_bpe.encode(raw_text)
print("Number of tokens in the encoded text:", len(encoded_text))

Number of tokens in the encoded text: 5145


In [58]:
encoded_sample= encoded_text[50:]

In [64]:
context_size = 4
x = encoded_sample[:context_size]
y = encoded_sample[1:context_size+1]
print(f"Input tokens (x): {x}")
print(f"Target tokens (y):      {y}")

Input tokens (x): [290, 4920, 2241, 287]
Target tokens (y):      [4920, 2241, 287, 257]


In [69]:
for i in range(1, context_size+1):
    context = encoded_sample[:i]
    desired = encoded_sample[i]
    # print(context, "--->", desired)
    print(tokenizer_bpe.decode(context), "--->", tokenizer_bpe.decode([desired]))

 and --->  established
 and established --->  himself
 and established himself --->  in
 and established himself in --->  a


In [81]:
import torch
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer_bpe, max_length, stride):
        self.input_ids =[]
        self.target_ids=[]

        token_ids =tokenizer_bpe.encode(txt)

        for i in range(0, len(token_ids)- max_length, stride):
            input_chunk = token_ids[i:i+max_length]
            target_chunk = token_ids[i+1:i+max_length+1]
            print(f"Input chunk: {tokenizer_bpe.decode(input_chunk)}")
            print(f"Target chunk: {tokenizer_bpe.decode(target_chunk)}")
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)
    
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [82]:
data_loader = GPTDatasetV1(raw_text, tokenizer_bpe, context_size, stride=3)

Input chunk: I HAD always
Target chunk:  HAD always thought
Input chunk:  always thought Jack G
Target chunk:  thought Jack Gis
Input chunk:  Gisburn rather
Target chunk: isburn rather a
Input chunk:  rather a cheap genius
Target chunk:  a cheap genius--
Input chunk:  genius--though a
Target chunk: --though a good
Input chunk:  a good fellow enough
Target chunk:  good fellow enough--
Input chunk:  enough--so it
Target chunk: --so it was
Input chunk:  it was no great
Target chunk:  was no great surprise
Input chunk:  great surprise to me
Target chunk:  surprise to me to
Input chunk:  me to hear that
Target chunk:  to hear that,
Input chunk:  that, in the
Target chunk: , in the height
Input chunk:  the height of his
Target chunk:  height of his glory
Input chunk:  his glory, he
Target chunk:  glory, he had
Input chunk:  he had dropped his
Target chunk:  had dropped his painting
Input chunk:  his painting, married
Target chunk:  painting, married a
Input chunk:  married a rich widow
Targe

In [83]:
def create_data_loader_v1(txt, batch_size = 4,max_length=256, 
                          stride=128, shuffle=True, drop_last=True, num_workers=0):

    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    data_loader = DataLoader(dataset,
                             batch_size=batch_size,
                             shuffle=shuffle,
                             drop_last=drop_last,
                             num_workers=num_workers
                             )
    return data_loader